In [162]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="3"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
from datasets import load_dataset
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import defaultdict


import prompts

In [46]:
## prepare data

# read evidence data
evidence_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/evidence_train.csv')
evidence_ls = evidence_df.loc[:,'text'].dropna().to_list()

#read qq data
qq_df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv')
qq_ds = Dataset.from_pandas(qq_df.loc[:,['question','follow_up_questions']])

In [47]:
qq_df

,id,sample_id,question,follow_up_questions
0,43810e53-3082-47c6-9a1c-7d530035252d,-5742327688291876861,When does episode 40 of bunk'd come out?,### When does episode 42 of bunk'd come out?\n...
1,11255093-2327-4d39-89e2-337be043db4a,-3582047784487750233,Who won the ncaa football national championshi...,### Who won the 2016 season's ncaa football na...
2,02b27ffc-f65a-45c1-a4c8-c3a90ecb4e59,6811938153834854976,"As of 2015, when was the last time the death p...","### As of 2017, when was the last time the dea..."
3,6b3a0668-67f9-421d-9ea7-a74abfafe252,1700733897006170137,Where does backwards failure of the left ventr...,### Where does failure of the left ventricle c...
4,c5db51fb-8310-4a12-bde2-9a4e17615094,142117929623619257,Who won the Second Italo-Ethiopian War?,### Who won the First Italo-Ethiopian War?\n##...
...,...,...,...,...
2882,fd3c54ec-f2c6-4911-9da6-fde28becc139,7007176913425291748,Where is wynonna earp filming for season 1 sup...,"### Where is wynonna earp, the tv series story..."
2883,92a74567-8ff9-483c-b5be-f794572c9b4a,1770991828175209436,When were the first fortifications built for t...,### When were the first fortifications built f...
2884,4ac0d29e-0508-4d9e-b1e9-73b3378bcbfd,-1402493466405577008,Who sang a cover of What You Won't Do for Love...,### Who sang the original What You Won't Do Fo...
2885,bb6969dd-70f2-406e-9a0f-dc4dfdda8c2c,3611220285690789892,When is my friend dahmer movie coming out in l...,### When is my friend dahmer movie coming out ...


In [48]:
qq_ds

Dataset({
    features: ['question', 'follow_up_questions'],
    num_rows: 2887
})

In [7]:
# load embedding model
model_path = '/raid/deallab/SF_RAG_Data/ASQA/models/fine_tuned_model_64'
model = SentenceTransformer(model_path)

# load model to device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# calculate embeddings for all data
evidence_embeddings = model.encode(evidence_ls, convert_to_tensor=True).to(device)

In [8]:
# load generative model
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")

quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=quantization_config,
    device_map="auto"  # 자동으로 적절한 디바이스 할당
)

tokenizer_gen.pad_token = tokenizer_gen.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gen.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps

In [163]:
importlib.reload(prompts)

<module 'prompts' from '/home/dataconv/deallab/djk/sf_rag/sf_rag/evaluate/prompts.py'>

In [166]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    query_embedding = model.encode(query, convert_to_tensor=True).to(device)

    query_embedding = query_embedding.unsqueeze(0)
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10]
    print(top_results)
    res={}
    for idx in top_results:
        tmp=evidence_ls[idx]
        res[tmp]=similarities[idx]
        
    return list(res.keys())

In [167]:
def evaluate_docs(query, docs):
    print(f"Query : {query}")
    print("-"*100)
    outs = []
    for idx, doc in enumerate(docs):
        #print(f"Rank {idx} : {doc}")
        
        input= f'''
        Query: {query}
        Doc:  {doc}
        '''

        messages = [
            {"role":"user", 'content':prompts.PROMPT['eval_doc_instr']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ1']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex2']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ2']},
            {"role":"user", 'content':prompts.PROMPT['eval_doc_ex3']},
            {"role":"assistant", 'content':prompts.PROMPT['eval_doc_answ3']},
            {"role":"user", 'content':input}, 
        ]
        #apply tokenizter + generate eval
        inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
        
        attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device)
        
        outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=128)
        generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
        
        filter=(generated_text.split('\n')[0])
        print(filter)
        if '#relevant' in filter:
            outs.append((generated_text.split('\n')[1]).strip())
    
    return outs

In [168]:
for entry in qq_ds:
    query = entry['question']
    fu_questions = entry['follow_up_questions']

In [169]:
def make_new_query(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['refine_query_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ1']},
        {"role":"user", 'content':prompts.PROMPT['refine_query_ex2']},
        {"role":"assistant", 'content':prompts.PROMPT['refine_query_answ2']},
        {"role":"user", 'content':{input}},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=256)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    return generated_text

In [143]:
qq_ds[0]

{'question': "When does episode 40 of bunk'd come out?",
 'follow_up_questions': "### When does episode 42 of bunk'd come out?\n### When does episode 41 of bunk'd come out?\n### When does episode 40 of bunk'd come out?"}

In [158]:
query = qq_ds[15]['question']
docs = retrieve_documents(query)
rel_docs = evaluate_docs(query, docs)

tensor([  1563,   1561,   1566,   1565,   1564,   1562,   1559, 172045, 176465,
        172044, 147343, 176473, 176476,  54700,  54699,  39543,  21562,  21650,
         38664,  65164], device='cuda:0')
Query : Who from New York City came up with the saying the customer is always right?
----------------------------------------------------------------------------------------------------
#irrelevant 
#irrelevant
#irrelevant
#irrelevant
#relevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#irrelevant
#relevant
#irrelevant


In [159]:
print(rel_docs)

["A representative of an unnamed New York company in 1909 stated that their policy was'regarding the customer as always right,' which contributed to building trade.", 'The remake of the film, which was a remake of the original 1947 film, involved a department store called "Cole\'s" in New York City, which was a replacement for Macy\'s, and also featured a fictional store called "Shopper\'s Express".']


In [146]:
def preprocessing(new_questions):
    return list(((new_questions.split("['")[1]).split("']")[0]).split("',\n    '"))

In [16]:
preprocessing(make_new_query(query, rel_docs))

['### Which teams played in the 2016 NCAA Football National Championship game and what were the scores?',
 '### What was the margin of victory for the Alabama Crimson Tide in the 2016 NCAA Football National Championship game?',
 '### Which team was defeated by the Alabama Crimson Tide in the Peach Bowl semifinal to advance to the National Championship game?',
 '### How many points did the Washington Huskies score in the 2016 NCAA Football National Championship game?',
 '### What was the score of the Peach Bowl semifinal game between the Alabama Crimson Tide and the Washington Huskies?',
 '### Which team did the Alabama Crimson Tide defeat in the Championship game to win the 2016 NCAA Football National Championship?']

In [170]:
def make_new_answer(query,context):
    
    input= f'''
    Original Query: {query}
    Context information: {context}
    '''
    
    messages = [
        {"role":"user", 'content':prompts.PROMPT['new_answer_instr']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ1']},
        {"role":"user", 'content':prompts.PROMPT['new_answer_ex2']},
        {"role":"assistant", 'content':prompts.PROMPT['new_answer_answ2']},
        {"role":"user", 'content':{input}},
    ]
    inputs = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)
    
    attention_mask = (inputs != tokenizer_gen.pad_token_id).long().to(device)
    
    outputs = model_gen.generate(inputs, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens=256)
    generated_text = tokenizer_gen.decode(outputs[0]).split('<|end_header_id|>')[-1].replace('<|eot_id|>', '').strip('\n')
    
    return generated_text

In [18]:
# import re

# def summarize_answers(question, answers):
#     input= f'''
#     Query: {query}
#     Context information: {answers}
#     '''
    
#     messages = [
#         {"role":"user", 'content':prompts.PROMPT['final_answer_instr']},
#         {"role":"assistant", 'content':prompts.PROMPT['final_answer_answ1']},
#         {"role":"user", 'content':{input}},
#     ]

#     #tokenizer prompt
#     input_ids = tokenizer_gen.apply_chat_template(messages, return_tensors="pt", truncation=True).to(device)

#     attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device)

#     out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
#     res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
#     candidate = [re.sub('\n|<\|eot_id\|>', '', res)]
#     return candidate

In [148]:
df=pd.read_csv('/raid/deallab/SF_RAG_Data/ASQA/qa_test.csv')

In [149]:
df.head()

,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,5cf39741-d5bd-4226-a94d-ff0c13ca5eaa,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,41196666-5ad7-4651-98aa-9fa9ddb4aad5,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,6b6a74ae-4b78-4a17-a338-5418ac1408cb,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,e94de938-6049-4e4f-9ac1-b6f98884e235,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,2943f760-b273-4602-bd66-6e5c73ae0edf,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [150]:
data=df[['question','long_answers']]
questions=data['question']

In [151]:
references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]

In [152]:
references[0]

{'id': '5cf39741-d5bd-4226-a94d-ff0c13ca5eaa',
 'sample_id': -7013890438520559398,
 'question': 'Who has the highest goals in world football?',
 'follow_up_questions': '["Who has the highest goals in men\'s world international football?", "Who has the highest goals all-time in men\'s football?", "Who has the highest goals in women\'s world international football?"]',
 'long_answers': '["Ali Dael has the highest goals in men\'s world international football with 109 goals. Josef Bican has the highest goals all-time in men\'s football and Christine Sinclair has the highest goals in women\'s world international football.", "The players with the highest all-time goals and highest men\'s and women\'s international football goals differ. The player with the highest all-time men\'s football goals is Josef Bican, who in 2020 was recognized by FIFA, the international governing body of football, as the record scorer with an estimated 805 goals. Christine Sinclair has the highest goals in women\'s

In [171]:
import re

def final_ans(query,answers):
    prompt = f"""
    Context information is below.
    ---------------------
    {answers}
    ---------------------
    Given the context information and not prior knowledge, 
    Answer questions that have multiple correct answers based on multiple interpretations, including multiple answers.
    Query: {query}
    Answer:
    """
    
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    candidate = [re.sub('\n|<\|eot_id\|>', '', res)]
    return candidate

In [182]:
from evaluation import evaluate

sf_rag=dict()
perplexity_df=pd.DataFrame()
scores_list=[]
new_answers_dic=dict()

for i in range(30):
    print(f"Query {i+1} : {questions[i]}")
    print("-"*100)
    query = questions[i]
    new_answers_dic[query]=list()
    docs = retrieve_documents(query)
    if rel_docs:
        new_questions=preprocessing(make_new_query(query, rel_docs))
    else:
        print("Pass to the next query")
        continue
    print(new_questions)
    for new_question in new_questions:
        new_docs=retrieve_documents(new_question)
        new_rel_docs=evaluate_docs(new_question, new_docs)
        if new_rel_docs:
            new_answers=make_new_answer(new_question, new_rel_docs)
            print(new_answers)
            new_answers_dic[query].append(new_answers)
        else:
            print('All irrelevant docs')
    candidate=final_ans(query,new_answers_dic[query])
    print(candidate)
    print(references[i])
    scores=evaluate(candidate,[references[i]])
    print(scores)
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    scores_df.mean()

Query 1 : Who has the highest goals in world football?
----------------------------------------------------------------------------------------------------
tensor([141763,  48264,  48248,  48265, 141709,  15415,  15401,  48246,  48247,
         48249], device='cuda:0')
['### Who are the top 101 goalscorers in the FIFA World Cup?', '### Which teams have had the most players score 100 goals or more in the FIFA World Cup?', '### Who is the record holder for most goals scored in a single tournament?', '### Who are the players that have achieved an average of 2 goals or more per match played in the FIFA World Cup?', '### Which players have scored for the most teams in the FIFA World Cup?', '### Who is the youngest player to score a goal in the FIFA World Cup?', '### Who is the oldest player to score a goal in the FIFA World Cup?', '### Who are the top 5 goalscorers in the FIFA World Cup?', '### What is the record for most goals scored in a single tournament?', '### Who holds the record for 

IndexError: list index out of range

In [174]:
scores_df=pd.DataFrame(scores_list)

In [175]:
scores_df.mean()

rougeLsum    29.178478
length       93.666667
str_em       30.555556
ovscore      18.200875
dtype: float64

In [176]:
len(scores_df)

6

In [87]:
scores_df.to_csv('/raid/deallab/SF_RAG_Data/ASQA/results/ours_results.csv', index=False)